# Usecase 3: CRC classification — run ritme models

Launches ritme classification experiments via the shared template `src/run_ritme_model.sh` and the `submit_model` helper in `src/launch_models.py`. Written against ritme >= 1.4.0. Set up the env once (the last command must be run from the repo root):

```shell
mamba create -n ritme_usecases -c adamova -c conda-forge -c bioconda -c pytorch ritme ipykernel nbconvert -y
conda activate ritme_usecases
pip install -e .
```

Two execution modes:
- `mode="slurm"` submits each model class as its own sbatch job using the SLURM resource defaults in `src/launch_models.py:MODEL_RESOURCES`.
- `mode="local"` runs the template inline in the current process — useful for short smoke tests.

The template performs (idempotently): train/test split (stratified by `srn`) → `find-best-model-config` → `evaluate-tuned-models` → bootstrap test-set CIs (classification metrics auto-dispatched in `src/bootstrap_metrics.py`) → `explain-features`.

## Setup

In [ ]:
from src.launch_models import submit_model

## Configuration

In [ ]:
# Where ritme writes experiment outputs. The launcher creates this dir
# (relative to the repo root if not absolute) and stores per-experiment
# subfolders + sbatch logs under it. Override for cluster runs, e.g.
# ``"/cluster/work/<lab>/<user>/ritme_runs_v150"``.
LOGS_DIR = "use_cases/ritme_runs/local"

# Cluster account (SLURM ``--account=...``, a.k.a. "share") forwarded to
# every sbatch submission below. ``None`` (default) lets each
# (usecase, model) row in ``SLURM_RESOURCES`` pick its own account: NN
# jobs auto-route to ``es_ilic`` (16 GPUs); everything else stays on the
# cluster default (es_beere). Set this to your own to override for ALL
# submissions in this notebook.
SLURM_ACCOUNT = None

## Launch experiments

Each call below submits one classifier model class. Replace `mode="local"` with `mode="slurm"` to dispatch as separate cluster jobs (using the per-model resources in `MODEL_RESOURCES`). Override `time_budget_s` etc. by editing the matching JSON in `config/`.

Excluded by design: `trac` (regression-only, squared-error loss), `nn_corn` (ordinal — the target is binary), and all regression-only models (`linreg`, `rf`, `xgb`, `nn_reg`).

In [ ]:
models = ["logreg", "rf_class", "xgb_class", "nn_class"]

common = dict(
    mode="slurm",
    logs_dir=LOGS_DIR,
    slurm_account=SLURM_ACCOUNT,
)
for model_type in models:
    submit_model("u3", model_type, **common)

## Launch experiments — without metadata enrichment (DEFERRED)

The cell below is preserved but commented out for this round. Re-enable
to run the no-enrich ablation; resources and time budget are inherited
unchanged from ``SLURM_RESOURCES``.


# models = ["logreg", "rf_class", "xgb_class", "nn_class"]
#
# for model_type in models:
#     submit_model('u3', model_type, variant="no_enrich", **common)


In [ ]:
models = ["logreg", "rf_class", "xgb_class", "nn_class"]

for model_type in models:
    submit_model("u3", model_type, variant="no_enrich", **common)

## Smoke test (optional)

Before submitting full-budget SLURM jobs, validate the classification dispatch end-to-end with a single short local run (overrides `time_budget_s` to 5 minutes):

In [ ]:
# submit_model(
#     "u3", "logreg", mode="local", logs_dir=LOGS_DIR,
#     config_overrides={"time_budget_s": 300},
# )